In [127]:
import pandas as pd

In [128]:
data = pd.read_csv('../../data/raw/real_transactions.csv')

In [129]:
data.head()

,Portfolio,Date,Transaction Type,Ticker,Quantity,Price,Fee
0,1,2023-03-10,BUY,VUAA.L,2,72.46,0.0
1,1,2023-04-11,BUY,VUAA.L,1,76.70,0.0
2,1,2023-05-10,BUY,VUAA.L,2,76.68,0.0
3,1,2023-06-09,BUY,VUAA.L,1,80.18,0.0
4,1,2023-07-14,BUY,VUAA.L,1,84.26,0.0


In [130]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Portfolio         78 non-null     int64  
 1   Date              78 non-null     str    
 2   Transaction Type  78 non-null     str    
 3   Ticker            78 non-null     str    
 4   Quantity          78 non-null     int64  
 5   Price             78 non-null     float64
 6   Fee               78 non-null     float64
dtypes: float64(2), int64(2), str(3)
memory usage: 4.4 KB


### Existence Checks

In [131]:
MUST_HAVE_COLUMNS = ["Portfolio", "Date", "Transaction Type", "Ticker", "Quantity", "Price", "Fee"]

missing_columns = []
for col in MUST_HAVE_COLUMNS:
    if col not in data.columns:
        missing_columns.append(col)

if len(missing_columns) > 0:
    raise ValueError(f"The following columns are missing from the data: {missing_columns}")

### Data Type Checks

In [132]:
MUST_HAVE_DATA_RULES = {
    "Portfolio": lambda x: (isinstance(x, str) and len(x) > 0) or isinstance(x, int),                   # int or str with length > 0
    "Date": lambda x: pd.to_datetime(x, format='%Y-%m-%d', errors='coerce') is not pd.NaT,              # format should match YYYY-MM-DD
    "Transaction Type": lambda x: x in ["BUY", "SELL"],                                                 # available values: BUY, SELL
    "Ticker": lambda x: isinstance(x, str) and len(x) > 0,                                              # str with length > 0
    "Quantity": lambda x: x > 0,                                                                        # positive number
    "Price": lambda x: x >= 0,                                                                          # non-negative number
    "Fee": lambda x: x >= 0                                                                             # non-negative number
}

rules_violated = []
for col in MUST_HAVE_DATA_RULES:
    if not all(data[col].apply(MUST_HAVE_DATA_RULES[col])):
        rules_violated.append(col)

if rules_violated:
    raise ValueError(f"Some values in columns {rules_violated} do not meet the required rules.")


MUST_HAVE_DATA_TYPES = {
    "Portfolio": str, 
    "Date": str,
    "Transaction Type": str,
    "Ticker": str,
    "Quantity": int,
    "Price": float,
    "Fee": float
}
data = data.astype(MUST_HAVE_DATA_TYPES)

data["Date"] = pd.to_datetime(data["Date"], format='%Y-%m-%d', errors='coerce')
data["Ticker"] = data["Ticker"].str.strip().str.upper()